# Lineborn — Strict Pre-Release Benchmark (Direct Colab)

This is the **direct benchmark notebook**. It does **not** connect to the Lineborn desktop app, does **not** open a Cloudflare tunnel, and does **not** require SIP credentials or a manually uploaded voice sample. Everything runs inside the temporary Colab VM.

It exercises the frozen pre-release AI/product gate across the shipping Lineborn model tiers: **Whisper small.en → Qwen → Chatterbox**, adversarial sales conversations, deterministic routing/safety checks, native tool behavior, text latency, STT accuracy, TTS real-time factor and composite voice-start latency.

### Run it
1. Choose **Runtime → Change runtime type → GPU**.
2. Run the single code cell below.
3. Let it finish without using the runtime for anything else.
4. Colab will download the canonical JSON report and the full results ZIP.

A failure is a release blocker for this gate. Do not loosen thresholds just to turn it green. Direct SIP/PSTN, carrier/NAT behavior, installer behavior, desktop integration and long campaign soak testing are separate gates.


In [ ]:
# ONE CLICK: strict direct-Colab pre-release benchmark
import json, pathlib, shutil, subprocess, sys
from google.colab import files

ROOT = pathlib.Path('/content')
REPO = ROOT / 'Axemetric-Caller-Beta-Runtime'

gpu = subprocess.run(
    ['nvidia-smi','--query-gpu=name,memory.total','--format=csv,noheader'],
    capture_output=True, text=True
)
if gpu.returncode != 0 or not gpu.stdout.strip():
    raise RuntimeError('No NVIDIA GPU detected. Choose Runtime > Change runtime type > GPU, reconnect, then rerun this cell.')
print('GPU(s):\n' + gpu.stdout.strip())

if REPO.exists():
    subprocess.run(['git','-C',str(REPO),'fetch','--depth','1','origin','main'], check=True)
    subprocess.run(['git','-C',str(REPO),'reset','--hard','origin/main'], check=True)
else:
    subprocess.run(['git','clone','--depth','1','https://github.com/SumamaAhmed69/Axemetric-Caller-Beta-Runtime.git',str(REPO)], check=True)

sha = subprocess.check_output(['git','-C',str(REPO),'rev-parse','HEAD'], text=True).strip()
print('\nBenchmark source commit:', sha)
print('Starting frozen Lineborn strict gate. First run installs the pinned AI stack and can take a while.\n')

bootstrap = REPO / 'benchmarks' / 'lineborn_strict_colab_bootstrap.py'
proc = subprocess.Popen(
    [sys.executable, str(bootstrap), '--models', 'all'],
    cwd=str(REPO / 'benchmarks'),
    stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True, bufsize=1
)
for line in proc.stdout:
    print(line, end='')
exit_code = proc.wait()

report_path = ROOT / 'lineborn-strict-prerelease' / 'lineborn-strict-prerelease.json'
zip_path = ROOT / 'lineborn-strict-prerelease-results.zip'
if not report_path.exists():
    raise RuntimeError(f'Benchmark exited with code {exit_code} without producing the canonical report.')

report = json.loads(report_path.read_text('utf-8'))
print('\n' + '=' * 80)
print('LINEBORN STRICT PRE-RELEASE:', 'PASS ✅' if report.get('release_gate_pass') else 'BLOCKED ❌')
print('=' * 80)
for model, data in (report.get('models') or {}).items():
    sales = data.get('sales') or {}
    tools = data.get('tools') or {}
    stream = data.get('stream_speed') or {}
    voice = data.get('voice_speed') or {}
    print(f"\n{model}: {'PASS' if data.get('passed') else 'FAIL'}")
    print('  Sales score      :', sales.get('product_score'))
    print('  Tool accuracy    :', tools.get('accuracy'))
    print('  TTFT median ms   :', stream.get('ttft_median_ms'))
    print('  Sentence p95 ms  :', stream.get('first_sentence_p95_ms'))
    print('  Voice median ms  :', voice.get('voice_start_median_ms'))
    print('  Voice p95 ms     :', voice.get('voice_start_p95_ms'))
    print('  STT WER median   :', voice.get('stt_wer_median'))
    print('  TTS RTF median   :', voice.get('tts_rtf_median'))
    failed = [k for k,v in (data.get('gate_checks') or {}).items() if not v]
    if failed:
        print('  FAILED GATES     :', ', '.join(failed))

print('\nDownloading canonical JSON and full ZIP...')
files.download(str(report_path))
if zip_path.exists():
    files.download(str(zip_path))

if exit_code not in (0, 2):
    raise RuntimeError(f'Benchmark process failed unexpectedly with exit code {exit_code}.')
